# Tutorial 3: Text Generation with Interventions

This notebook teaches you how to generate text with mlxterp while applying interventions (steering vectors, ablations) at every generation step.

**What you'll learn:**
- Basic text generation (greedy, temperature, top-k, top-p)
- Applying interventions during generation
- Using callbacks for monitoring and custom stopping
- Comparing normal vs intervened generation
- Working with `GenerationResult`

**Prerequisites:** Basic familiarity with mlxterp (see [01_causal_patching.ipynb](01_causal_patching.ipynb)).

## 1. Setup

In [ ]:
from mlxterp import InterpretableModel
from mlxterp.core.intervention import zero_out, scale, add_vector, noise
import mlx.core as mx

model = InterpretableModel("mlx-community/Llama-3.2-1B-Instruct-4bit")
print(f"Model loaded: {len(model.layers)} layers")

## 2. Basic Generation

The simplest case: generate text with greedy decoding (temperature=0).

In [ ]:
# Greedy generation — always picks the most likely token
result = model.generate(
    "The capital of France is",
    max_tokens=20,
    temperature=0.0,
)

print(f"Prompt: 'The capital of France is'")
print(f"Generated: '{result.text}'")
print(f"Tokens: {result.tokens}")
print(f"Token count: {len(result.tokens)}")

In [ ]:
# Greedy is deterministic — running again gives the same result
result2 = model.generate("The capital of France is", max_tokens=20, temperature=0.0)
assert result.tokens == result2.tokens, "Greedy should be deterministic!"
print("✓ Greedy generation is deterministic")

## 3. Sampling Strategies

Temperature, top-k, and top-p control the randomness and diversity of generation.

In [ ]:
prompt = "Once upon a time in a land far away"

# Temperature sampling — higher = more creative/random
for temp in [0.0, 0.3, 0.7, 1.0]:
    result = model.generate(prompt, max_tokens=30, temperature=temp)
    print(f"temp={temp}: {result.text[:80]}...")

In [ ]:
# Top-k sampling — only consider the k most likely tokens
result = model.generate(
    prompt,
    max_tokens=30,
    temperature=0.8,
    top_k=40,  # Only sample from the top 40 tokens
)
print(f"Top-k (k=40): {result.text[:100]}")

In [ ]:
# Top-p (nucleus) sampling — keep tokens until cumulative prob exceeds p
result = model.generate(
    prompt,
    max_tokens=30,
    temperature=0.8,
    top_p=0.95,  # Keep tokens covering 95% of probability mass
)
print(f"Top-p (p=0.95): {result.text[:100]}")

In [ ]:
# Combine strategies
result = model.generate(
    prompt,
    max_tokens=30,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
)
print(f"Combined (t=0.7, k=50, p=0.9): {result.text[:100]}")

## 4. Generation with Interventions

The key feature: apply interventions at **every** generation step. This lets you study how modifications affect the model's extended behavior, not just a single forward pass.

In [ ]:
# Ablation: zero out a specific MLP during generation
normal = model.generate("The president of the United States is", max_tokens=15)
ablated = model.generate(
    "The president of the United States is",
    max_tokens=15,
    interventions={"layers.10.mlp": zero_out},
)

print(f"Normal:  {normal.text}")
print(f"Ablated: {ablated.text}")

In [ ]:
# Scale down attention — reduce its influence
normal = model.generate("The capital of France is", max_tokens=15)
scaled = model.generate(
    "The capital of France is",
    max_tokens=15,
    interventions={"layers.5.self_attn": scale(0.1)},
)

print(f"Normal (attn full):     {normal.text}")
print(f"Scaled (attn × 0.1):   {scaled.text}")

In [ ]:
# Add noise to see how robust the model is
normal = model.generate("2 + 2 =", max_tokens=10)
noisy = model.generate(
    "2 + 2 =",
    max_tokens=10,
    interventions={"layers.8": noise(std=0.5)},
)

print(f"Normal: {normal.text}")
print(f"Noisy:  {noisy.text}")

In [ ]:
# Multiple interventions simultaneously
result = model.generate(
    "Hello, my name is",
    max_tokens=20,
    interventions={
        "layers.3.mlp": scale(0.5),
        "layers.7.self_attn": scale(2.0),  # Amplify attention
    },
)
print(f"Multi-intervention: {result.text}")

## 5. Steering Vectors

Construct a steering vector from contrastive examples and use it during generation.

In [ ]:
# Create a simple steering vector from contrastive prompts
# Positive: happy sentiment, Negative: sad sentiment
layer_idx = 8

with model.trace("I am very happy and joyful today") as trace:
    pass
positive_act = None
for key in trace.activations:
    if f"layers.{layer_idx}" in key and 'resid_post' in key:
        positive_act = trace.activations[key]
        break
if positive_act is None:
    for key in trace.activations:
        if key.endswith(f"layers.{layer_idx}"):
            positive_act = trace.activations[key]
            break

with model.trace("I am very sad and miserable today") as trace:
    pass
negative_act = None
for key in trace.activations:
    if f"layers.{layer_idx}" in key and 'resid_post' in key:
        negative_act = trace.activations[key]
        break
if negative_act is None:
    for key in trace.activations:
        if key.endswith(f"layers.{layer_idx}"):
            negative_act = trace.activations[key]
            break

if positive_act is not None and negative_act is not None:
    mx.eval(positive_act, negative_act)
    # Steering vector = mean difference at last token
    steering_vec = positive_act[0, -1, :] - negative_act[0, -1, :]
    mx.eval(steering_vec)
    print(f"Steering vector shape: {steering_vec.shape}")
    print(f"Steering vector norm: {float(mx.sqrt(mx.sum(steering_vec ** 2))):.2f}")
else:
    print("Could not extract activations — adjust layer_idx or check model")

In [ ]:
# Generate with positive and negative steering
prompt = "I think the weather today is"

if positive_act is not None and negative_act is not None:
    normal = model.generate(prompt, max_tokens=25, temperature=0.7)
    positive = model.generate(
        prompt, max_tokens=25, temperature=0.7,
        interventions={f"layers.{layer_idx}": add_vector(steering_vec * 3.0)},
    )
    negative = model.generate(
        prompt, max_tokens=25, temperature=0.7,
        interventions={f"layers.{layer_idx}": add_vector(steering_vec * -3.0)},
    )
    
    print(f"Normal:   {normal.text}")
    print(f"Positive: {positive.text}")
    print(f"Negative: {negative.text}")

## 6. Callbacks and Stop Tokens

In [ ]:
# Monitor generation step by step
def monitor_callback(step, token_id, logits):
    token_str = model.tokenizer.decode([token_id])
    top_prob = float(mx.max(mx.softmax(logits)))
    print(f"  Step {step:2d}: '{token_str}' (confidence: {top_prob:.1%})")
    return False  # Don't stop

print("Generating with step-by-step monitoring:")
result = model.generate(
    "The answer to life is",
    max_tokens=10,
    callback=monitor_callback,
)
print(f"\nFull output: {result.text}")

In [ ]:
# Custom stopping: stop at first period
def stop_at_period(step, token_id, logits):
    token_str = model.tokenizer.decode([token_id])
    return '.' in token_str

result = model.generate(
    "The Eiffel Tower was built in",
    max_tokens=50,
    callback=stop_at_period,
)
print(f"Stopped at period: {result.text}")

In [ ]:
# Using stop_tokens
result = model.generate(
    "List three colors:",
    max_tokens=50,
    stop_tokens=[model.tokenizer.eos_token_id],
)
print(f"Generated (stopped at EOS): {result.text}")

## 7. Working with GenerationResult

In [ ]:
result = model.generate("Machine learning is", max_tokens=15)

# Summary
print(result.summary())
print()

# Structured data
print(f"Text: '{result.text}'")
print(f"Tokens: {result.tokens}")
print(f"Prompt: '{result.prompt}'")
print(f"Has logits: {result.token_logits is not None}")
if result.token_logits is not None:
    print(f"Logits shape: {result.token_logits.shape}")
print()

# JSON export
import json
parsed = json.loads(result.to_json())
print(f"Result type: {parsed['result_type']}")
print(f"Data keys: {list(parsed['data'].keys())}")

In [ ]:
# Analyze per-token logits
if result.token_logits is not None:
    print("Per-token confidence:")
    for i, (tid, logits) in enumerate(zip(result.tokens, result.token_logits)):
        token_str = model.tokenizer.decode([tid])
        prob = float(mx.softmax(logits)[tid])
        print(f"  Token {i}: '{token_str}' — probability: {prob:.1%}")

## 8. Input Formats

Generation accepts multiple input types.

In [ ]:
# String input (requires tokenizer)
r1 = model.generate("Hello", max_tokens=5)
print(f"String input: {r1.text}")

# Token ID list
token_ids = model.tokenizer.encode("Hello")
r2 = model.generate(token_ids, max_tokens=5)
print(f"Token list input: {r2.text}")

# mx.array
r3 = model.generate(mx.array([token_ids]), max_tokens=5)
print(f"mx.array input: {r3.text}")

## Summary

| Feature | Code | Purpose |
|---------|------|---------|
| Greedy generation | `temperature=0.0` | Deterministic, most likely sequence |
| Temperature | `temperature=0.8` | Control randomness |
| Top-k | `top_k=40` | Limit token choices |
| Top-p | `top_p=0.95` | Nucleus sampling |
| Interventions | `interventions={...}` | Modify behavior during generation |
| Stop tokens | `stop_tokens=[eos_id]` | Stop at specific tokens |
| Callbacks | `callback=fn` | Monitor or custom stop |

**Next:** Try [04_conversation_analysis.ipynb](04_conversation_analysis.ipynb) to learn about multi-turn conversation analysis.